#**Problem Statement:**

A retail store that has multiple outlets across the country are facing issues in managing the inventory - to match the demand with respect to supply.

**Dataset Information:**

The walmart.csv contains 6435 rows and 8 columns.


Feature Name -  Description

Store -  Store number

Date -  Week of Sales

Weekly_Sales -  Sales for the given store in that week

Holiday_Flag  - If it is a holiday week

Temperature -  Temperature on the day of the sale

Fuel_Price  - Cost of the fuel in the region

CPI -  Consumer Price Index

Unemployment  - Unemployment Rate


1. You are provided with the weekly sales data for their various outlets. Use statistical analysis, EDA, outlier analysis, and handle the missing values to come up with various
insights that can give them a clear perspective on the following:

a. If the weekly sales are affected by the unemployment rate, if yes - which stores are suffering the most?

b. If the weekly sales show a seasonal trend, when and what could be the reason?

c. Does temperature affect the weekly sales in any manner?

d. How is the Consumer Price index affecting the weekly sales of various stores?

e. Top performing stores according to the historical data.

f. The worst performing store, and how significant is the difference between the
highest and lowest performing stores.


2. Use predictive modeling techniques to forecast the sales for each store for the next 12 weeks.

**Dataset**

https://www.google.com/url?q=https%3A%2F%2Fdrive.google.com%2Ffile%2Fd%2F1NudGzvfLGgM8_9aMb03AtKKGCADG7aSM%2Fview


**Load Data**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv('../data/raw/walmart.csv')
df.head()

In [ ]:
df.tail()

**EDA**

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.dtypes

In [ ]:
# change Date datatype to datetime
df['Date']=pd.to_datetime(df['Date'], format='%d-%m-%Y')

In [ ]:
df.dtypes

In [ ]:
df.head()

In [ ]:
df.describe().T

In [ ]:
# outliers
for col in df.columns:
  if df[col].dtype == 'int64' or df[col].dtype == 'float64':
    sns.boxplot(df[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

**Insight:**

Outliers are present but plausibly real, so keeping them is reasonable


In [ ]:
# distribution
num_col = df.select_dtypes(include=['int64','float64']).columns


for col in num_col:
  plt.figure(figsize=(6,4))
  sns.histplot(df[col], kde=True)
  plt.title(f"Distribution of {col}")
  plt.grid(True)
  plt.show()

**Insights:**



In [ ]:
# correlation heatmap for numeric columns
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()


**1. Provide clear perspective on the following:**

**a.  If the weekly sales are affected by the unemployment rate, if yes - which stores are suffering the most?**

In [ ]:
# 1. overall correlation between Weekly_Sales and Unemployment

df[['Weekly_Sales', 'Unemployment']].corr()

In [ ]:
# 2. scatter plot visual analysis

plt.figure(figsize=(10,6))
sns.scatterplot(x='Unemployment', y='Weekly_Sales', data=df)
plt.title('Weekly Sales vs Unemployment')
plt.show()

In [ ]:
# 3. store-level correlation analysis

store_corr = df.groupby('Store').apply(
    lambda x: x['Weekly_Sales'].corr(x['Unemployment'])
).reset_index(name='Correlation')

store_corr = store_corr.sort_values(by='Correlation')
store_corr.head()

In [ ]:
# 4. Identify worst stores

worst_affected = store_corr.nsmallest(5, 'Correlation')
worst_affected

In [ ]:
# barplot visualization

plt.figure(figsize=(10,6))
sns.barplot(x='Store', y='Correlation', data=store_corr)
plt.xticks(rotation=90)
plt.title("Store-wise Correlation: Sales vs Unemployment")
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
# 5. regression analysis - statistcal validation

import statsmodels.api as sm

X = df['Unemployment']
y = df['Weekly_Sales']

X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())

**Impact of Umemployment on Weekly Sales**

1. Overall Relationship

The correlation between Weekly_Sales and Unemploymnet is:

-0.106

This indicates a weak negative relationship, suggesting that as unemployment increases, weekly sales tend to decrease slightly.

However, the strenght of this relationship is very low, implying that unemployment is noty a primary driver of sales at the aggregate level.

2. Scatter plot Visual Analysis

The scatter plot of Weekly_Sales vs Unemployment shows:

*  A wide dispersion of data points

*  No strong linear pateern

*  Significant variability in sales across similar unemployment levels

This reinforces that the relationship between unemployment and sales is weak and likely influenced by other factors such as store location, seasonality, and promotions.

3. Store-level Analysis (Key Insight)

To uncover deeper patterns, a store-level correlation analysis was performed.

This revealed that the impact of unemployment is not uniform across all stores.

Most Affected Stores (Strong Neghative Correlation)

Store, Correlation

38, -0.785

44, -0.780

39, -0.384

42, -0.356

41, -0.350

Interpretation:

*  These stores show a strong inverse relationship between unemployment and sales

*  As unemployment rises, sales in these stores drop significantly

4. Stores with Positive Correlation (Interesting Finding)

From bar plot visualization, some stores (e.g., Store 36) show strong positive correlation (~0.83).

Possible explanation:

*  Regional economic differences

*  Essential goods demand remains stable or increases

*  Data anomalies or external factors (e.g store size, other promotions)

5. Regression Analysis (Statistical Validation)

Asimple linear regression model was fitted:

Coefficient (Unemployment): ~31,940

P-value: < 0.001 (statistically insignificant)

R2: 0.011

Interpretation:

The relationship is statistically significant, meaning it is not due to random chance

However, the very low R2 (1.1%) indicates that unemployment explains only a small portion of sales variability.

**Final Business Insight**

While unemployment has a statistically significant negative effect on weekly sales, its overall impact is weak at the aggregate level.

However:

Certain stores (notably Stores 38,44,39,42,and 41) are highly sensitive to unemployment changes, indicating localized economic dependence.

**Business Recommendation**

*  Focus demand planning and inventory strategies on highly sensitive stores

*  During periods of rising unemployment:
  
  *  Adjust inventory downward for affected stores

  *  Introduce targeted promotions or discounts

*  Incorporate regional economic indicators into forecasting models



**b. If the weekly sales show a seasonal trend, when and what could be the reason?**

In [ ]:
df.head()

In [ ]:
# 1. Sort df by Date

df_sorted=df.sort_values(by='Date')
df_sorted.head()

In [ ]:
# 2. aggregate total weekly sales


weekly_sales_total=df_sorted.groupby('Date')['Weekly_Sales'].sum().reset_index()
weekly_sales_total.head()

In [ ]:
# 3. plot time series

plt.figure(figsize=(15,7))
plt.plot(weekly_sales_total['Date'], weekly_sales_total['Weekly_Sales'])
plt.title("Total Weekly Sales Over Time")
plt.xlabel("Date")
plt.ylabel("Weekly Sales")
plt.grid(True)
plt.show()

In [ ]:
# 4. extract time features

df_sorted['Year'] = df_sorted['Date'].dt.year
df_sorted['Month'] = df_sorted['Date'].dt.month
df_sorted['Week']= df_sorted['Date'].dt.isocalendar().week

In [ ]:
df_sorted.head()

In [ ]:
# 5. monthly seasonality

monthly_sales = df_sorted.groupby('Month')['Weekly_Sales'].mean()

monthly_sales.plot(kind='bar', figsize=(10,6), title="Average Sales by Month")
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
# 6. holiday impact

holiday_sales = df_sorted.groupby('Holiday_Flag')['Weekly_Sales'].mean()
holiday_sales

In [ ]:
# visualization

sns.boxplot(x='Holiday_Flag', y='Weekly_Sales', data=df_sorted)
plt.title("Sales Distribution: Holiday vs Non-Holiday")
plt.show()

In [ ]:
# 7. year-over-year pattern

plt.figure(figsize=(14,6))

for year in df_sorted['Year'].unique():
  temp = df_sorted[df_sorted['Year']==year]
  temp = temp.groupby('Week')['Weekly_Sales'].mean()
  plt.plot(temp.index, temp.values, label=year)

plt.legend()
plt.title("Weekly Sales Pattern by Year")
plt.xlabel("Week Number")
plt.ylabel("Average Sales")
plt.show()


**Seasonal Trend Analysis of Weekly Sales**

**1. Overall Trend**

A time series analysis of total weekly sales reveals clear seasonal patterns across the years.

*   Sales exhibit recurring spikes at specific periods
*   These peaks are consistent across multiple years, indicating strong seasonality

**2. Monthly Seasonality**

Analysis of average monthly sales shows:

*   Highest sales occur in November and December
*   Lowest sales occur in January and early February
*   Mid-year sales remain relatively stable

This pattern aligns with:

*   Holiday shopping (Black Friday, Christmas)
*   Post-holiday spending decline

**3. Holiday Impact**

Sales during holiday weeks are significantly higher than non-holiday weeks.

*  Holiday periods show elevated average sales
*  Increased variability due to promotional events and consumer demand spikes

**4. Year-over-Year Weekly Patterns**

Weekly sales trends across different years show consistent peaks around the same weeks.

This confirms the presence of strong annual seasonality

**Final Business Insight**

Weekly sales exhibit strong seasonal behavior driven primarily by holiday periods and yearly retail cycles.

Key observations:

*  Sales surge significantly during Q4 (Nov–Dec)
*  Demand drops sharply after holidays
*  Patterns repeat consistently year-over-year

**Business Recommendation**

*  Increase inventory and staffing ahead of holiday seasons

*  Optimize supply chain for Q4 demand spikes

*  Use seasonality features in forecasting models

*  Plan promotions strategically around known peak periods


**c. Does temperature affect the weekly sales in any manner?**

In [ ]:
# 1. correlation

df[['Weekly_Sales', 'Temperature']].corr()

In [ ]:
# 2. scatter plot

sns.scatterplot(x='Temperature', y='Weekly_Sales', data=df)
plt.title("Weekly Sales vs Temperature")
plt.show()

**Impact of Temperature on Weekly Sales**

The correlation between temperature and weekly sales is weak (0.06381), indicating no strong linear relationship.

*  Sales occur across a wide range of temperatures
*  No consistent increase or decrease pattern is observed

**Business Insight:**

Temperature does not significantly influence weekly sales. Consumer purchasing behavior appears to be driven more by seasonal events (holidays) rather than weather conditions.


**d. How is the Consumer Price index affecting the weekly sales of various stores?**

In [ ]:
# 1. correlation analysis

df[['Weekly_Sales', 'CPI']].corr()

In [ ]:
# 2. scatter plot of CPI vs weekly sales

sns.scatterplot(x='CPI', y='Weekly_Sales', data=df)
plt.title("Weekly Sales vs CPI")
plt.show()

**Impact of Consumer Price Index (CPI) on Weekly Sales**

The analysis shows a weak relationship between CPI and weekly sales.

*  CPI changes gradually over time, while sales fluctuate weekly
*  No strong or consistent pattern is observed between the two variables

**Business Insight:**

CPI does not have a direct short-term impact on weekly sales. However, it may influence long-term purchasing power and should be considered in long-term forecasting models rather than short-term analysis.


**e. Top performing stores according to the historical data.**

In [ ]:
store_performance = df.groupby('Store')['Weekly_Sales'].sum().reset_index()
top_stores = store_performance.sort_values(by='Weekly_Sales', ascending=False)
top_stores.head()

**Top Performing Stores**

The top-performing stores were identified based on total historical sales.

*  These stores consistently generate the highest revenue
*  They are key contributors to overall business performance

**Business Insight**

High-performing stores should be prioritized for:

*  inventory allocation
*  marketing campaigns
*  demand forecasting optimization

**The worst performing store, and how significant is the difference between the highest and lowest performing stores.**

In [ ]:
# analysis

worse_stores = store_performance.sort_values(by='Weekly_Sales').head()
print("Worst Performing Stores:")
print(worse_stores)

max_sales = store_performance['Weekly_Sales'].max()
min_sales = store_performance['Weekly_Sales'].min()

difference = max_sales - min_sales
print("Difference between highest and lowest performing store:", difference)

**Worst Performing Stores and Performance Gap**

The lowest-performing stores were identified based on total sales.

*  These stores contribute significantly less revenue compared to top-performing locations

The difference between the highest and lowest performing stores is substantial, indicating a large performance gap across the retail network.

**Business Insight**

This disparity suggests:

*  regional demand differences
*  store size/location impact
*  potential inefficiencies in certain locations


Based on EDA, sales are primarily driven by seasonality and store-level patterns rather than macroeconomic or environmental variables.

This directly justifies:

*  time-based features
*  lag features
*  per-store modeling

**2. Use predictive modeling techniques to forecast the sales for each store for the next 12 weeks.**

In [ ]:
# model building
# 1. choose one Store - Store 1

store_1 = df[df['Store']==1].copy()
store_1 = store_1.sort_values(by='Date')
store_1.set_index('Date', inplace=True)

ts = store_1['Weekly_Sales']
ts.head()

In [ ]:
# 2. visualize the store time series

plt.figure(figsize=(14,6))
plt.plot(ts)
plt.title('Store 1 Weekly Sales')
plt.xlabel('Date')
plt.ylabel("Weekly Sales")
plt.grid(True, alpha=0.2)
plt.show()

*  trend
*  seasonality
*  spikes
*  changing variance

In [ ]:
# 3. split train/test by time
# since I need to forecast the next 12 weeks, I will reserve the last 12 weeks as test data

train = ts.iloc[:-12]
test = ts.iloc[-12:]

print(train.shape, test.shape)

In [ ]:
# 4. run ADF on training data

from statsmodels.tsa.stattools import adfuller

def adf_test(series, title=''):
  result = adfuller(series.dropna())
  print(f'ADF Test: {title}')
  print(f'ADF Statistic: {result[0]}')
  print(f'p-value: {result[1]}')
  print(f'# Lags Used: {result[2]}')
  print(f'# Observations: {result[3]}')
  for key, value in result[4].items():
    print(f'Critical Value ({key}): {value}')
  if result[1] < 0.05:
    print("Result: Reject null hypothesis -> series is stationary")
  else:
    print("Result: Fail to reject null hypothesis -> series is non-stationary")

adf_test(train, 'Store 1 Weekly Sales')

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

seasonal_decompose(train, period=52).plot();

**Time Series Decomposition (Store 1)**

To better understand the underlying structure of the time series, a seasonal decomposition was performed.

**1. Trend Component**

The decomposition reveals a gradual upward trend in weekly sales over time.

*  This trend is not immediately obvious in the raw time series due to noise
*  However, it indicates a steady increase in baseline sales

**2. Seasonal Component**

The seasonal component shows repeating patterns, particularly:

*  Strong spikes during Q4 (November–December)
*  These correspond to major retail events such as Black Friday and Christmas

**3. Residual Component**
*  Residuals are centered around zero, indicating that most systematic patterns have been captured
*  Remaining spikes likely represent irregular events or promotions

**4. Interpretation**

The time series exhibits both a mild upward trend and clear seasonal behavior, making it suitable for models that can capture both components.


In [ ]:
# 5. modeling implications
# model choice to handle Trend, Seasonality, and Noise

**Model choice -> SARIMA**

Because:

*  Captures trend(d)

*  Captures seasonality (D, s)
*  Handles autoregression + noise

In [ ]:
# acf/ pacf

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.figure(figsize=(12,5))
plot_acf(train, lags=40)
plt.show()

plt.figure(figsize=(12,5))
plot_pacf(train, lags=40)
plt.show()

**ACF and PACF Analysis**

The ACF and PACF plots were analyzed to determine appropriate model parameters.

**1. ACF Observations**

*  Significant spike at lag 1

*  Gradual decay afterward

*  Indicates the presence of autoregressive behavior

**2. PACF Observations**

*  Strong cutoff at lag 1

*  Minor contributions from lag 2

**3. Interpretation**

The combination of ACF tailing off and PACF cutting off at lag 1 suggests an AR(1) process, with a possible extension to AR(2).

**4. Modeling Implication**

Non-seasonal ARIMA models will be tested with:

*  p = 1 or 2
*  d = 0 or 1
*  q = 0 or 1

Seasonal SARIMA models will also be evaluated to capture holiday-driven patterns, using:

*  seasonal period = 52 (weekly data)


In [ ]:
# 6. fit ARIMA baseline

from statsmodels.tsa.arima.model import ARIMA

model_arima = ARIMA(train, order=(1,0,0))
fit_arima = model_arima.fit()

forecast_arima = fit_arima.forecast(steps=12)

In [ ]:
# fit SARIMA

from statsmodels.tsa.statespace.sarimax import SARIMAX

model_sarima = SARIMAX(
    train,
    order=(1,0,0),
    seasonal_order=(1,1,0,52),
    enforce_stationarity=False,
    enforce_invertibility=False
)

fit_sarima = model_sarima.fit(disp=False)

forecast_sarima = fit_sarima.forecast(steps=12)

In [ ]:
# 7. evaluate

from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_forecast(actual, pred, model_name='Model'):
    actual = np.array(actual)
    pred = np.array(pred)

    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100

    print(f'{model_name} MAE: {mae:,.2f}')
    print(f'{model_name} RMSE: {rmse:,.2f}')
    print(f'{model_name} MAPE: {mape:.2f}%')

    return mae, rmse, mape

In [ ]:
evaluate_forecast(test, forecast_arima, 'ARIMA')
evaluate_forecast(test, forecast_sarima, 'SARIMA')

In [ ]:
# forecast visualization

plt.figure(figsize=(14,6))
plt.plot(train.index, train, label='Train')
plt.plot(test.index, test, label='Test')
plt.plot(test.index, forecast_arima, label='ARIMA Forecast')
plt.plot(test.index, forecast_sarima, label='SARIMA Forecast')

plt.title('ARIMA vs SARIMA Forecast')
plt.xlabel('Date')
plt.ylabel('Weekly Sales')
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

**Model Comparison: ARIMA vs SARIMA**

Two classical time series models were evaluated on Store 1 using the final 12 weeks as the test set.

**Forecast Accuracy Results**

Model,      MAE,	            RMSE,	       MAPE,

ARIMA	    57,719.44	         67,692.26	   3.72%

SARIMA	  66,143.84	         75,056.85	   4.22%

**Interpretation**

*  ARIMA outperformed SARIMA across all evaluation metrics
*  Although seasonal patterns were observed in the exploratory analysis, incorporating seasonal structure did not improve short-term forecast accuracy for Store 1
*  This suggests that, for this store, recent autoregressive behavior was more informative than the seasonal component over the 12-week forecast horizon


**Conclusion**

ARIMA was selected as the preferred forecasting model for Store 1 because it produced the lowest forecast error on the holdout period.

**Important modeling insight**

*  seasonality exists in the broader dataset
*  but not every store benefits equally from a seasonal model
*  therefore, model performance should be validated store by store, rather than assuming SARIMA is always best

**I will test a few more ARIMA and SARIMA close variants**

In [ ]:
# ARIMA variants to test

# ARIMA(1,0,0) was already tested, so we will try other variants
model_arima_200 = ARIMA(train, order=(2,0,0))
fit_arima_200 = model_arima_200.fit()
forecast_arima_200 = fit_arima_200.forecast(steps=12)

model_arima_101 = ARIMA(train, order=(1,0,1))
fit_arima_101 = model_arima_101.fit()
forecast_arima_101 = fit_arima_101.forecast(steps=12)

model_arima_201 = ARIMA(train, order=(2,0,1))
fit_arima_201 = model_arima_201.fit()
forecast_arima_201 = fit_arima_201.forecast(steps=12)

In [ ]:
# SARIMA variants to test

#SARIMAX(train, order=(1,0,1), seasonal_order=(1,1,0,52))
model_sarima_101 = SARIMAX(
    train,
    order=(1,0,1),
    seasonal_order=(1,1,0,52),
    enforce_stationarity=False,
    enforce_invertibility=False
)

fit_sarima_101 = model_sarima_101.fit(disp=False)
forecast_sarima_101 = fit_sarima_101.forecast(steps=12)


#SARIMAX(train, order=(1,0,0), seasonal_order=(1,0,0,52))
model_sarima_100 = SARIMAX(
    train,
    order=(1,0,0),
    seasonal_order=(1,0,0,52),
    enforce_stationarity=False,
    enforce_invertibility=False
)

fit_sarima_100 = model_sarima_100.fit(disp=False)
forecast_sarima_100 = fit_sarima_100.forecast(steps=12)

In [ ]:
# evaluate and compare models

def evaluate_forecast(actual, pred, model_name='Model'):
    actual = np.array(actual)
    pred = np.array(pred)

    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100

    return {
        'Model': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape
    }

forecast_dict = {
    'ARIMA(1,0,0)': pd.Series(forecast_arima, index=test.index),
    'ARIMA(2,0,0)': pd.Series(forecast_arima_200, index=test.index),
    'ARIMA(1,0,1)': pd.Series(forecast_arima_101, index=test.index),
    'ARIMA(2,0,1)': pd.Series(forecast_arima_201, index=test.index),
    'SARIMA(1,0,0)(1,1,0,52)': pd.Series(forecast_sarima, index=test.index),
    'SARIMA(1,0,0)(1,0,0,52)': pd.Series(forecast_sarima_100, index=test.index),
    'SARIMA(1,0,1)(1,1,0,52)': pd.Series(forecast_sarima_101, index=test.index)
}

results = []

for model_name, forecast in forecast_dict.items():
    result = evaluate_forecast(test, forecast, model_name)
    results.append(result)

results_df = pd.DataFrame(results).sort_values(by='RMSE').reset_index(drop=True)

print(results_df)

print(f"\nBest model: {results_df.loc[0, 'Model']}")
print(f"Best RMSE: {results_df.loc[0, 'RMSE']:,.2f}")
print(f"Best MAPE: {results_df.loc[0, 'MAPE']:.2f}%")



In [ ]:
# visualize forecast comparism across top 3 models

plt.figure(figsize=(14,6))
plt.plot(train.index, train, label='Train')
plt.plot(test.index, test, label='Test', linewidth=2)

# Get the names of the top 3 models
top_3_model_names = results_df.head(3)['Model'].tolist()

# Plot only the top 3 models
for model_name in top_3_model_names:
    plt.plot(test.index, forecast_dict[model_name], label=model_name)

plt.title('Forecast Comparison Across Top 3 Models')
plt.xlabel('Date')
plt.ylabel('Weekly Sales')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.2)
plt.show()

**Model Comparison and Selection (Store 1)**

Multiple ARIMA and SARIMA models were evaluated using MAE, RMSE, and MAPE on a 12-week holdout set.

| Model                       | MAE    | RMSE       | MAPE      |
| --------------------------- | ------ | ---------- | --------- |
| **SARIMA(1,0,0)(1,0,0,52)** | 31,892 | **38,618** | **2.05%** |
| SARIMA(1,0,1)(1,1,0,52)     | 38,965 | 45,946     | 2.52%     |
| ARIMA(1,0,0)                | 57,719 | 67,692     | 3.72%     |

**Key Findings**

*  SARIMA models significantly outperformed ARIMA models
*  The best-performing model was SARIMA(1,0,0)(1,0,0,52)
*  This model achieved the lowest RMSE and MAPE

**Interpretation**

The superior performance of SARIMA confirms that seasonality plays a critical role in weekly sales patterns, even when not strongly visible in ACF/PACF plots.

*  Seasonal structure (period = 52 weeks) captures recurring yearly demand patterns
*  ARIMA models fail to capture this repeating behavior

**Final Model Selection**

SARIMA(1,0,0)(1,0,0,52) was selected as the optimal model for Store 1 due to its superior forecasting accuracy.

**Business Insight**

Sales patterns are strongly influenced by annual seasonal cycles, particularly holiday-driven demand.

Implications:

*  Inventory planning must account for yearly demand spikes
*  Forecasting models must include seasonal components
*  Ignoring seasonality leads to significant prediction error


In [ ]:
# 8. scale for all 45 Stores
df.head()

In [ ]:
# prepare data

df = df.sort_values(['Store', 'Date'])

In [ ]:
# define evaluation function

def evaluate_forecast(actual, pred):
    actual = np.array(actual)
    pred = np.array(pred)

    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100

    return mae, rmse, mape

In [ ]:
# create output containers

# A. store-level model performance
# B. future 12-week forecasts per store

store_metrics = []
future_forecasts = []

In [ ]:
df['Store'].unique()

In [ ]:
# automation to loop throuigh all stores

for store_id in sorted(df['Store'].unique()):
    try:
        # Filter one store
        store_df = df[df['Store'] == store_id].copy()
        store_df = store_df.sort_values('Date')
        store_df.set_index('Date', inplace=True)
        store_df = store_df.asfreq('W-FRI')

        ts = store_df['Weekly_Sales']

        # Skip stores with too little data
        if len(ts) < 60:
            continue

        # Train-test split
        train = ts.iloc[:-12]
        test = ts.iloc[-12:]

        # Fit on training data
        model = SARIMAX(
            train,
            order=(1, 0, 0),
            seasonal_order=(1, 0, 0, 52),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fit = model.fit(disp=False)

        # Forecast on test period
        test_forecast = fit.forecast(steps=12)
        test_forecast = pd.Series(test_forecast, index=test.index)

        # Evaluate
        mae, rmse, mape = evaluate_forecast(test, test_forecast)

        # Save metrics
        store_metrics.append({
            'Store': store_id,
            'MAE': mae,
            'RMSE': rmse,
            'MAPE': mape
        })

        # Refit on full history
        final_model = SARIMAX(
            ts,
            order=(1, 0, 0),
            seasonal_order=(1, 0, 0, 52),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        final_fit = final_model.fit(disp=False)

        # Forecast next 12 future weeks
        future_pred = final_fit.forecast(steps=12)

        # Build future dates
        last_date = ts.index.max()
        future_dates = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=12, freq='W-FRI')

        # Save future forecasts
        store_forecast_df = pd.DataFrame({
            'Store': store_id,
            'Date': future_dates,
            'Forecasted_Weekly_Sales': future_pred.values
        })

        future_forecasts.append(store_forecast_df)

    except Exception as e:
        print(f"Store {store_id} failed: {e}")

In [ ]:
# convert results into dataframes

metrics_df = pd.DataFrame(store_metrics).sort_values('RMSE').reset_index(drop=True)
future_forecasts_df = pd.concat(future_forecasts, ignore_index=True)

In [ ]:
# view model performance across stores
# shows best forecasted stores by error

metrics_df.head(10)

In [ ]:
# summary of overall performance

metrics_df.describe().T

In [ ]:
# view sample future forecasts

future_forecasts_df.head(20)

In [ ]:
# plot forecast for one store - Store 45

store_id = 45

store_df = df[df['Store'] == store_id].copy()
store_df = store_df.sort_values('Date')
store_df.set_index('Date', inplace=True)
store_df = store_df.asfreq('W-FRI')

ts = store_df['Weekly_Sales']
future_store = future_forecasts_df[future_forecasts_df['Store'] == store_id]

plt.figure(figsize=(14,6))
plt.plot(ts.index, ts, label='Historical Sales')
plt.plot(future_store['Date'], future_store['Forecasted_Weekly_Sales'], label='12-Week Forecast')

plt.title(f'Store {store_id}: Historical Sales and 12-Week Forecast')
plt.xlabel('Date')
plt.ylabel('Weekly Sales')
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
# save outputs

metrics_df.to_csv('store_forecasting_metrics.csv', index=False)
future_forecasts_df.to_csv('store_12_week_forecasts.csv', index=False)

**Forecasting Weekly Sales for All Stores**

After identifying SARIMA(1,0,0)(1,0,0,52) as the optimal model for Store 1, the same modeling approach was automated across all 45 stores to generate 12-week forecasts.

**Modeling workflow**

For each store:

*  weekly sales data was sorted by date
*  the final 12 observed weeks were reserved as a test set
*  the model was trained on the remaining history
*  forecast accuracy was evaluated using MAE, RMSE, and MAPE
*  the model was then refit on the full store history to generate forecasts for the next 12 weeks

**Model Performance Summary**

Average RMSE across stores: ~72,482

Average MAPE: ~6.20%

Best-performing store MAPE: ~1.58%

Worst-performing store MAPE: ~17.98%

**Key Findings**

1. Strong overall forecasting performance
*  Most stores achieved low prediction error
*  The model performs well for short-term forecasting (12 weeks)

2. Significant variation across stores
*  Some stores are highly predictable
*  Others show higher volatility and forecasting difficulty
3. Seasonality is a dominant factor
*  The success of SARIMA confirms that annual seasonal cycles (Q4 demand spikes) strongly influence sales

**Business Implications**

**1. Inventory Planning**

*  Highly predictable stores allow for precise inventory allocation
*  Volatile stores require safety stock and flexible planning

**2. Demand Forecasting Strategy**
*  Store-level modeling is essential
*  A single global model would not capture store-specific patterns

**3. Seasonal Preparation**
*  Retailers must prepare for significant demand increases during holiday periods
*  Forecasting models must incorporate seasonal components

**Deliverables**

The final output includes:

**1. Store-level forecasting performance metrics**

**2. 12-week future sales forecasts for each store**

These output can be used for:

*  demand planning
*  inventory optimization
*  staffing decisions

**A useful extra analysis**

In [ ]:
# best forecasted stores - top 10

metrics_df.nsmallest(10, 'RMSE')

In [ ]:
# hardest stores to forecast

metrics_df.nlargest(10, 'RMSE')

In [ ]:
df.to_csv('historical_sales.csv', index=False)